In [1]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
import datetime
from OSDE.LegendreExpSPDensity import LegExp, LegExpSPDensity, LegExpResult
from StocProcess.RBM import RBMTransProb, MakeRBMTransProbFunc
from QAE.RQAE import RQAE

In [2]:
# RBM parameters
c = -1
d = 1
x0 = 0.5 * (c + d)
t0 = 0
mu = 0.5
sigma = 1.0
n_terms = 5

tN = 0.6
t1 = 0.2

# approximation setting
maxDeg = 10
R = 12
eps0 = 1 / 2**10
integEpsabs = 1e-4
nRep = 10

In [3]:
np.random.seed(1)

In [4]:
Ns = np.tile((2 ** np.linspace(3, 6.5, 8)).astype(int), nRep)
print(Ns)

[ 8 11 16 22 32 45 64 90  8 11 16 22 32 45 64 90  8 11 16 22 32 45 64 90
  8 11 16 22 32 45 64 90  8 11 16 22 32 45 64 90  8 11 16 22 32 45 64 90
  8 11 16 22 32 45 64 90  8 11 16 22 32 45 64 90  8 11 16 22 32 45 64 90
  8 11 16 22 32 45 64 90]


In [5]:
# PDF at time t
transProbFunc = MakeRBMTransProbFunc(tN, t0, c, d, mu, sigma, n_terms)

# Prob(X > (c + d)/2)
integFunc = lambda x: transProbFunc(x, x0)
pTrue, _ = sp.integrate.quad(integFunc, x0, 1.0)

In [6]:
epss = []
pEsts = []
totalQueryNums = []
maxDepths = []
NsComp = []

for N in Ns:
    legExpResultPrev = None
    ts = np.concatenate([[t0], np.linspace(t1, tN, N)])
    eps = eps0 / np.sqrt(N)
    epss.append(eps)
    totalQueryNum = 0
    maxDepth = 0
    print(datetime.datetime.now(), "N=", N)

    for i in range(len(ts)-1):
        print("i_t=", i, datetime.datetime.now())
        transProbFunc = MakeRBMTransProbFunc(ts[i+1], ts[i], c, d, mu, sigma, n_terms)

        if i == 0:
            densFunc = lambda x: transProbFunc(x, x0)
            legExpResult = LegExp(densFunc, maxDeg)
        else:
            legExpResult = LegExpSPDensity(legExpResultPrev.fApp, transProbFunc, maxDeg, epsabs=integEpsabs)

        coefs = np.zeros(maxDeg+1)
        coefs[0] = 0.5

        for l in range(1, maxDeg+1):
            a = 0.5 * (legExpResult.coefs[l] / (l + 0.5) + 1)
            rqaeResult = RQAE(a, eps, R)
            coefs[l] = (2 * rqaeResult.aEst -1) * (l + 0.5)
            totalQueryNum += rqaeResult.TotalQueryNum
            maxDepth = max(maxDepth, rqaeResult.MaxDepth)

        legExpResultPrev = LegExpResult(coefs)

    pEsts.append(sp.integrate.quad(legExpResultPrev.fApp, x0, 1.0)[0])
    totalQueryNums.append(totalQueryNum)
    maxDepths.append(maxDepth)
    NsComp.append(N)

    retDf = pd.DataFrame(dict(N=NsComp,
                              pTrue=np.repeat(pTrue, len(NsComp)),
                              eps=epss,
                              pEst=pEsts,
                              absErr=np.abs(np.array(pEsts) - pTrue),
                              totalQueryNum=totalQueryNums,
                              maxDepth=maxDepths))
    retDf.to_csv('DivideRBM_RQAE.csv', index=False)

2025-02-14 18:26:45.327467 N= 8
i_t= 0 2025-02-14 18:26:45.329469


c:\Users\koich\Desktop\Code\DivQCOSDE\QAE\MaximizeL.py:11: RuntimeWarning: divide by zero encountered in log
  neglogL = lambda theta: -np.dot(n1s, np.log(np.sin(thetaMuls * theta)**2)) - np.dot(n0s, np.log(np.cos(thetaMuls * theta)**2))


i_t= 1 2025-02-14 18:26:47.880086
i_t= 2 2025-02-14 18:27:11.724834
i_t= 3 2025-02-14 18:27:36.227606
i_t= 4 2025-02-14 18:28:04.447871
i_t= 5 2025-02-14 18:28:32.646836
i_t= 6 2025-02-14 18:28:56.919201
i_t= 7 2025-02-14 18:29:17.913896
2025-02-14 18:29:38.887276 N= 11
i_t= 0 2025-02-14 18:29:38.887276
i_t= 1 2025-02-14 18:29:41.747689
i_t= 2 2025-02-14 18:30:13.683836
i_t= 3 2025-02-14 18:30:38.900121
i_t= 4 2025-02-14 18:31:07.957483
i_t= 5 2025-02-14 18:31:36.889884
i_t= 6 2025-02-14 18:32:06.426086
i_t= 7 2025-02-14 18:32:31.720696
i_t= 8 2025-02-14 18:33:00.166224
i_t= 9 2025-02-14 18:33:28.922357
i_t= 10 2025-02-14 18:33:53.934446
2025-02-14 18:34:18.827629 N= 16
i_t= 0 2025-02-14 18:34:18.827629
i_t= 1 2025-02-14 18:34:21.555395
i_t= 2 2025-02-14 18:35:03.662124
i_t= 3 2025-02-14 18:35:45.984198
i_t= 4 2025-02-14 18:36:24.450914
i_t= 5 2025-02-14 18:37:02.980483
i_t= 6 2025-02-14 18:37:30.641116
i_t= 7 2025-02-14 18:38:02.362659
i_t= 8 2025-02-14 18:38:37.658866
i_t= 9 2025-02-

KeyboardInterrupt: 

In [ ]:
retDf

,N,pTrue,eps,pEst,absErr,totalQueryNum,maxDepth
0,8,0.649605,0.000345,0.649584,0.000021,5286272,2895


In [ ]:
retDf.groupby('N')['absErr'].mean()

N
8      0.003221
11     0.002877
16     0.002774
22     0.002475
32     0.002233
45     0.001315
64     0.002383
90     0.001689
128    0.003843
Name: absErr, dtype: float64